# rng-throughput
This notebook measures the throughput of random number generation as implemented by the C++ standard library.

```python
for dist in distributions:
    for gen in generators:
        tic()
        for i in range(n_iters):
            x = dist(gen)

        time_s = toc()
        throughput = sizeof(x) * n_iters / time_s
```

The benchmark can be implemented with the python language and benefits from:
  * string interpolation
  * JIT compilation
  * LLVM passes to detect *dead code elimination*

In [1]:
# usual llvm initialization
import llvmlite
import llvmlite.binding as llvm

from peppo.opt import usual_optimize
from peppo.ext_source import ExtSource, Language

llvm.initialize_native_target()
llvm.initialize_native_asmprinter()

## Domain specific code
Users of the PEPPO framework declare the program logic using primitive data and functions.
The user code is ingested with the `ExtSource` class, which is responsible of compiling the code into LLVM IR.

Once an LLVM module is created, the limited functionality (as exposed by the `llvmlite` dependency) can be accessed.
This involves optimization passes and JIT compilation, but in principle custom passes can be added.

In [2]:
DISTRIBUTIONS = [
    'normal_distribution<double>',
    'uniform_real_distribution<double>'
]

GENERATORS = [
    'minstd_rand0',
    'minstd_rand',
    'mt19937',
    'mt19937_64',
    'ranlux24_base',
    'ranlux48_base',
    'ranlux24',
    'ranlux48',
    'knuth_b',
]

In [3]:
def generate_benchmark(entry_point: str,
                     distribution: str,
                     generator: str) -> ExtSource:
    # digraphs `<%` are used to not escape the curly braces
    src = f"""
    #include <chrono>
    #include <random>

    using namespace std;

    double {entry_point}(std::size_t nruns) <%
        {distribution} dist;
        {generator} gen;

        const auto t_start = std::chrono::steady_clock::now();
        for (; nruns; --nruns) <%
            const auto x = dist(gen);
        %>
        const auto t_end = std::chrono::steady_clock::now();

        const std::chrono::duration<double> delta = t_end - t_start;
        return delta.count();
    %>
    """

    return ExtSource(src, Language.CPP)

## Exploration with the PEPPO framework

In [4]:
target = llvm.Target.from_default_triple()
target_machine = target.create_target_machine()

In [5]:
ENTRY_POINT = 'profile_generation'
benchmarks = {}
optimized_benchmarks = {}

for dist in DISTRIBUTIONS:
    for gen in GENERATORS:
        bench_src = generate_benchmark(
            ENTRY_POINT,
            dist,
            gen
        )

        benchmarks[(dist, gen)] = bench_src.compile_to_llvm_ir()

In [6]:
for key, module in benchmarks.items():
    optimized_benchmarks[key] = usual_optimize(
        target_machine,
        module,
        speed_level=2
    )

### Simple analysis pass
We are trying to prevent the dead code elimination for the `profile_generation` function.
The simplest solution would be to detect if there is still a loop after the optimization pass.

In [7]:
from peppo.utils import get_closest_function_name

In [8]:
def detect_dce_pass(module: llvm.ModuleRef) -> bool:
    bench_function_name = get_closest_function_name(
        module,
        ENTRY_POINT
    )[0]
    func = module.get_function(bench_function_name)

    nblocks = 0
    for block in func.blocks:
        nblocks += 1

    return nblocks

In [9]:
for key, module in optimized_benchmarks.items():
    print(key, detect_dce_pass(module))

('normal_distribution<double>', 'minstd_rand0') 3
('normal_distribution<double>', 'minstd_rand') 3
('normal_distribution<double>', 'mt19937') 5
('normal_distribution<double>', 'mt19937_64') 5
('normal_distribution<double>', 'ranlux24_base') 5
('normal_distribution<double>', 'ranlux48_base') 7
('normal_distribution<double>', 'ranlux24') 5
('normal_distribution<double>', 'ranlux48') 7
('normal_distribution<double>', 'knuth_b') 5
('uniform_real_distribution<double>', 'minstd_rand0') 8
('uniform_real_distribution<double>', 'minstd_rand') 8
('uniform_real_distribution<double>', 'mt19937') 10
('uniform_real_distribution<double>', 'mt19937_64') 10
('uniform_real_distribution<double>', 'ranlux24_base') 10
('uniform_real_distribution<double>', 'ranlux48_base') 12
('uniform_real_distribution<double>', 'ranlux24') 10
('uniform_real_distribution<double>', 'ranlux48') 12
('uniform_real_distribution<double>', 'knuth_b') 10


In [10]:
curr = list(optimized_benchmarks.values())[7]

In [11]:
print(curr)

; ModuleID = '<string>'
source_filename = "-"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128"
target triple = "x86_64-pc-linux-gnu"

%"class.std::normal_distribution.43" = type <{ %"struct.std::normal_distribution<>::param_type.44", double, i8, [7 x i8] }>
%"struct.std::normal_distribution<>::param_type.44" = type { double, double }
%"class.std::discard_block_engine.45" = type { %"class.std::subtract_with_carry_engine.46", i64 }
%"class.std::subtract_with_carry_engine.46" = type { [12 x i64], i64, i64 }

$_ZNSt19normal_distributionIdEclISt20discard_block_engineISt26subtract_with_carry_engineImLm48ELm5ELm12EELm389ELm11EEEEdRT_RKNS0_10param_typeE = comdat any

$_ZNSt20discard_block_engineISt26subtract_with_carry_engineImLm48ELm5ELm12EELm389ELm11EEclEv = comdat any

; Function Attrs: mustprogress uwtable
define dso_local noundef double @_Z18profile_generationm(i64 noundef %0) local_unnamed_addr #0 {
  %2 = alloca %"class.std::normal_di

In [12]:
func = detect_dce_pass(curr)
print(func)

7
